# miniGPT-DDP on Kaggle 2x T4

**Setup**: Settings sidebar -> Accelerator -> **GPU T4 x2**. Internet ON.

Upload the project files (`model.py`, `data.py`, `train.py`, `benchmark.py`) as a Kaggle Dataset, attach it, and the first cell copies them into the working directory.

**Session survival**: checkpoints land in `/kaggle/working/checkpoints/`. Before a session dies (12h limit), `Save Version` persists outputs; next session, attach the previous version's output as input and copy the checkpoint back to resume.

In [ ]:
# copy project files from the attached dataset (adjust path to your dataset name)
!cp /kaggle/input/mini-gpt-ddp/*.py /kaggle/working/
%cd /kaggle/working
!pip install -q tiktoken datasets
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# one-time: tokenize TinyStories (~15 min, CPU). Skip if data/ is attached from a prior version.
!python data.py --num_proc 4

In [ ]:
# resume support: if a previous version's checkpoint is attached, copy it back
import os, shutil, glob
os.makedirs('checkpoints', exist_ok=True)
for ckpt in glob.glob('/kaggle/input/*/checkpoints/*.pt'):
    shutil.copy(ckpt, 'checkpoints/')
    print('restored', ckpt)

In [ ]:
# benchmark FIRST (cheap, ~2 min total): the scaling-efficiency numbers
!torchrun --standalone --nproc_per_node=1 benchmark.py
!torchrun --standalone --nproc_per_node=2 benchmark.py
# bonus: sync-cost curve -- efficiency vs gradient accumulation
!torchrun --standalone --nproc_per_node=2 benchmark.py --grad_accum_steps 1
!torchrun --standalone --nproc_per_node=2 benchmark.py --grad_accum_steps 4

In [ ]:
# main training run (resumes automatically if a checkpoint exists)
# ~6000 iters at ~0.5M tokens/iter = ~3B tokens; expect this to span sessions
!torchrun --standalone --nproc_per_node=2 train.py --run_name ddp_2gpu

In [ ]:
# for the writeup: identical config on ONE gpu -- wall-clock comparison
# (run in a separate session so it doesn't eat the main run's quota)
# !torchrun --standalone --nproc_per_node=1 train.py --run_name single_gpu --max_iters 1000